## ref.prep_platformEOM

Migration of on-prem stored procedure `[PLTFRM].[ref].[prep_platformEOM]` to Databricks.

**Purpose:** Monthly platform-level exposure snapshot for subsidy-eligible loans.

### Parameters
| Parameter | Description | Default |
| --- | --- | --- |
| `run_date` | Override base date (YYYY-MM-DD). Leave empty for auto derivation from `ref_calendar`. | `""` (auto) |

### Source Tables
| Cloud Table | Filter | Role |
| --- | --- | --- |
| `{env}-cdh.landed.rpt_applicationsubsidy` | max BusinessDate | Pool (subsidy) |
| `{env}-cdh.landed.profits_lns_ext_subsidy_hd` | max BusinessDate | Pool (PROFITS) |
| `{env}-cdh.derivative.get_all_exposures_by_business_date` | base\_date | Loan attributes (all exposures incl. closed) |
| `{env}-cdh.primary.exposure` | latest SCD per key | Loan-attribute fallback for inactive loans not in function |
| `{env}-cdh.derivativeactive.dexposure` | base\_date | Financial attributes (active only) |
| `{env}-cdh.derivativeactive.dcollection` | base\_date | Collection payments |
| `{env}-cdh.primaryactive.paymentschedule` | plan\_date | Next-month installments |

### Target Table
`{env}-cdh-delivery.it.ref_platformEOM` — full overwrite per run.

### Architecture

**Dual-source exposure join:**
* **Function** (`get_all_exposures_by_business_date`): loan-level attrs (IsActiveFlag, MaturityDate, MechanismCode, InstalmentFrequency) for all exposures incl. closed. Mirrors on-prem `dwh.ast.loan` with `currentRecord=1`.
* **Fallback** (`primary.exposure` latest SCD): recovers \~948 inactive loans whose AccountNumber has no ExposureKey match in the function. Loan attrs are static — historical snapshot matches on-prem at 99.8%+. Used via `COALESCE(function, fallback)`.
* **dexposure** (`derivativeactive.dexposure`): financial fields (ReferenceDate, OTB, LoanStatus, InterestRate, Delinquent, PastDueDays, BucketId) for active loans only. Mirrors on-prem `LoanFinancials` SCD date filter. NULL for closed loans — this is correct and matches on-prem behaviour.

**Key design decisions:**
* `InstalmentFrequency` 0→1 conversion applied: on-prem maps freq=0 to `"1"` for most loans.
* `DaysPastDueFinal` used (not raw `DaysPastDue`) to match on-prem.
* `OutstandingTotalBalance` cast to string to match on-prem output format.

### Known Limitations

* **LoanId**: 0% match by design — cloud uses ExposureKey, on-prem uses FLDT internal ID.
* **BucketId (low Total%)**: On-prem BucketId is 100% non-null (sources it separately, likely `ISNULL(LoanFinancials.BucketCode, Loan.BucketCode)`). Cloud dexposure has no equivalent fallback for closed loans. Historical dexposure tested but too stale. Not fixable with current cloud data.
* **PaymentAmount (low NonNull%)**: On-prem join routes through `dwh.ast.asset` (captures re-mapped loanids); cloud joins `dcollection.ExposureKey` directly. Structural data-mapping difference, not a code bug.
* **installFrq (\~6.6% non-null gap)**: Two causes: (1) on-prem inconsistency on freq=0 (349 loans map to `"1"`, 285 keep `"0"`); (2) fallback SCD drift for \~948 recovered loans.
* **Financial fields NULL for closed loans**: dexposure has no rows for inactive exposures. Alternatives investigated (COALESCE with function, historical dexposure) were not viable — function financials are ALL NULL, historical snapshots are stale. Current implementation is the most accurate with available cloud sources.

In [0]:
import sys
from datetime import date, datetime
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F

try:
    sys.path.append("/Workspace/Shared")
    import aws_utils
except ImportError as e:
    raise RuntimeError(f"Failed to import aws_utils from /Workspace/Shared: {e}") from e

env = aws_utils.get_env_prefix()
TARGET_TABLE = f"`{env}-cdh-delivery`.it.`ref_platformEOM`"

# Base date: from widget or auto from ref_calendar
run_date_param = dbutils.widgets.get("run_date").strip()

if run_date_param:
    try:
        _bd = datetime.strptime(run_date_param, "%Y-%m-%d").date()
    except ValueError:
        raise ValueError(f"run_date must be YYYY-MM-DD, got: '{run_date_param}'")
    base_date = str(_bd)
else:
    row = (
        spark.table(f"`{env}-cdh`.reference.ref_calendar")
        .filter(F.col("CalendarDate") == F.current_date().cast("string"))
        .select("PreviousMonthLastWorkingDate")
        .first()
    )
    if row is None:
        raise RuntimeError("No calendar row found for today — check ref_calendar")
    base_date = str(row["PreviousMonthLastWorkingDate"])

# Derived dates from base_date
_bd = datetime.strptime(base_date, "%Y-%m-%d").date()
plan_date = str(_bd - relativedelta(months=1))
start_dt  = str((_bd + relativedelta(months=1)).replace(day=1))
_sd       = datetime.strptime(start_dt, "%Y-%m-%d").date()
end_dt    = str(_sd + relativedelta(months=1) - relativedelta(days=1))
prv_start = str(_sd - relativedelta(months=1))
prv_end   = str(_sd - relativedelta(days=1))


# Latest available BusinessDate per source
_r1 = spark.table(f"`{env}-cdh`.landed.rpt_applicationsubsidy").agg(F.max("BusinessDate")).collect()[0][0]
_r2 = spark.table(f"`{env}-cdh`.landed.profits_lns_ext_subsidy_hd").agg(F.max("BusinessDate")).collect()[0][0]
if _r1 is None or _r2 is None:
    raise RuntimeError(f"Missing BusinessDate: rpt_applicationsubsidy={_r1}, profits={_r2}")
max_bd_subsidy = str(_r1)
max_bd_profits = str(_r2)


print(f"env={env}  target={TARGET_TABLE}")
print(f"base_date={base_date}  plan_date={plan_date}")
print(f"start_dt={start_dt}  end_dt={end_dt}  prv_start={prv_start}  prv_end={prv_end}")
print(f"max_bd_subsidy={max_bd_subsidy}  max_bd_profits={max_bd_profits}")

In [0]:
# Source 1: rpt_applicationsubsidy
df_src1 = (
    spark.table(f"`{env}-cdh`.landed.rpt_applicationsubsidy")
    .filter(
        (F.col("BusinessDate") == F.lit(max_bd_subsidy)) &
        (F.col("new_account_number").isNotNull()) &
        (F.col("CurrentRecord") == 1) &
        (F.col("cepal_setEligibiltyDate").isNotNull()) &
        (F.col("PlatformName").isNotNull())
    )
    .select(
        F.lower("PlatformName").alias("platformName"),
        F.col("ApplicationNumber").alias("applicationNumber"),
        F.col("DebtNumber").alias("debtNumber"),
        F.col("new_account_number").alias("AccountNumber"),
        F.lit(None).cast("string").alias("settlementCode")
    )
)

# Source 2: profits_lns_ext_subsidy_hd (DEBT_TYPE → platformName)
df_src2 = (
    spark.table(f"`{env}-cdh`.landed.profits_lns_ext_subsidy_hd")
    .filter(F.col("BusinessDate") == F.lit(max_bd_profits))
    .select(
        F.when(F.col("DEBT_TYPE") == "3", "bridge2")
         .when(F.col("DEBT_TYPE") == "1", "bridge1")
         .when(F.col("DEBT_TYPE") == "2", "n4605")
         .when(F.col("DEBT_TYPE") == "4", "bridge3")
         .alias("platformName"),
        F.col("APPLICATION_NO").alias("applicationNumber"),
        F.col("DEBT_NUMBER").alias("debtNumber"),
        F.col("ACCOUNT_NUMBER").alias("AccountNumber"),
        F.lit(None).cast("string").alias("settlementCode")
    )
)

# Source 3: Bridge3 subset
df_src3 = (
    spark.table(f"`{env}-cdh`.landed.rpt_applicationsubsidy")
    .filter(
        (F.col("BusinessDate") == F.lit(max_bd_subsidy)) &
        (F.col("PlatformName") == "Bridge3")
    )
    .select(
        F.lit("bridge3").alias("platformName"),
        F.col("ApplicationNumber").alias("applicationNumber"),
        F.col("DebtNumber").alias("debtNumber"),
        F.col("new_account_number").alias("AccountNumber"),
        F.lit(None).cast("string").alias("settlementCode")
    )
)

# UNION + distinct
df_pool = df_src1.unionByName(df_src2).unionByName(df_src3).distinct()

pool_count = df_pool.count()
if pool_count == 0:
    raise RuntimeError(
        f"Pool is empty (rpt_applicationsubsidy BD={max_bd_subsidy}, "
        f"profits BD={max_bd_profits}). Check source tables."
    )
print(f"Pool count: {pool_count:,}")

In [0]:
# Function: all exposures incl. closed
df_func = (
    spark.sql(f"""
        SELECT * FROM `{env}-cdh`.derivative.get_all_exposures_by_business_date('{base_date}')
        WHERE PrimaryPartyFlag = true
    """)
)

# dexposure: financial attrs, active loans only (NULL for closed)
df_dexposure = (
    spark.table(f"`{env}-cdh`.derivativeactive.dexposure")
    .filter(
        (F.col("BusinessDate") == F.lit(base_date)) &
        (F.col("PrimaryPartyFlag") == True)
    )
)

# dcollection: matched payments in previous month
df_payments = (
    spark.table(f"`{env}-cdh`.derivativeactive.dcollection")
    .filter(
        (F.col("BusinessDate") == F.lit(base_date)) &
        (F.col("ActualDate").between(F.lit(prv_start), F.lit(base_date))) &
        (F.col("MatchedDate").isNotNull()) &
        (F.col("IsPaymentFlag") == True)
    )
    .groupBy("ExposureKey")
    .agg(
        F.max("ActualDate").alias("paymentDate"),
        F.sum("Amount").alias("PaymentAmount")
    )
)

# Fallback: latest SCD from primary.exposure for loans not in function
from pyspark.sql.window import Window

_pe_cols = (
    spark.table(f"`{env}-cdh`.primary.exposure")
    .filter(F.col("PrimaryPartyFlag") == True)
    .select("ExposureKey", "BusinessDate", "IsActiveFlag", "Maturitydate",
            "MechanismCode", "InstalmentFrequency")
)

df_exp_fallback = (
    df_pool.select("AccountNumber").distinct()
    .join(_pe_cols, F.col("AccountNumber") == F.col("ExposureKey"), "inner")
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("ExposureKey").orderBy(F.col("BusinessDate").desc())
    ))
    .filter(F.col("_rn") == 1)
    .select(
        F.col("ExposureKey").alias("fb_Key"),
        F.col("IsActiveFlag").alias("fb_IsActiveFlag"),
        F.col("Maturitydate").alias("fb_Maturitydate"),
        F.col("MechanismCode").alias("fb_MechanismCode"),
        F.col("InstalmentFrequency").alias("fb_InstalmentFrequency"),
    )
)

# Join all sources
df_pool_loan = (
    df_pool.alias("p")
    .join(
        df_func.alias("f"),
        F.col("p.AccountNumber") == F.col("f.ExposureKey"),
        "left"
    )
    .join(
        df_exp_fallback.alias("fb"),
        F.col("p.AccountNumber") == F.col("fb.fb_Key"),
        "left"
    )
    .join(
        df_dexposure.alias("e"),
        F.col("p.AccountNumber") == F.col("e.ExposureKey"),
        "left"
    )
    .join(
        df_payments.alias("pay"),
        F.col("p.AccountNumber") == F.col("pay.ExposureKey"),
        "left"
    )
    .select(
        F.col("p.platformName"),
        F.col("p.applicationNumber"),
        F.col("p.debtNumber"),
        F.col("p.AccountNumber"),
        F.col("p.settlementCode").alias("settlementcode"),
        F.col("p.AccountNumber").alias("AssetKey"),
        # Loan attrs: COALESCE(function, fallback)
        F.coalesce(F.col("f.ExposureKey"), F.col("fb.fb_Key")).alias("LoanId"),
        F.coalesce(F.col("f.IsActiveFlag"), F.col("fb.fb_IsActiveFlag")).alias("isactiveflag"),
        F.coalesce(F.col("f.Maturitydate"), F.col("fb.fb_Maturitydate")).alias("MaturityDate"),
        F.when(
            F.coalesce(F.col("f.InstalmentFrequency"), F.col("fb.fb_InstalmentFrequency")) == 0,
            F.lit("1")
        ).otherwise(
            F.coalesce(F.col("f.InstalmentFrequency"), F.col("fb.fb_InstalmentFrequency")).cast("string")
        ).alias("installFrq"),
        F.coalesce(F.col("f.MechanismCode"), F.col("fb.fb_MechanismCode")).alias("accMechanism"),
        # Financials from dexposure (NULL for closed loans)
        F.col("e.BusinessDate").alias("ReferenceDate"),
        F.col("e.OutstandingTotalBalance").cast("string").alias("OutstandingTotalBalance"),
        F.col("e.ExposureStatusCode").alias("LoanStatus"),
        F.col("e.InterestRate"),
        F.col("e.DueAmount").alias("Delinquent"),
        F.col("e.DaysPastDueFinal").alias("PastDueDays"),
        F.col("e.BucketCode").cast("int").alias("BucketId"),
        # Payments
        F.col("pay.PaymentAmount"),
        F.col("pay.paymentDate").alias("paymentdate"),
    )
)

pool_loan_count = df_pool_loan.count()
if pool_loan_count != pool_count:
    raise RuntimeError(
        f"PoolLoan count {pool_loan_count:,} != Pool count {pool_count:,} — "
        f"left join produced duplicates or lost rows. Check for duplicate ExposureKeys in dexposure."
    )
print(f"PoolLoan count: {pool_loan_count:,}")

In [0]:
# Next-month installments

df_payment_schedule = (
    spark.table(f"`{env}-cdh`.primaryactive.paymentschedule")
    .filter(
        (F.col("BusinessDate") == F.lit(plan_date)) &
        (F.col("SourceSystem") == "profits")
    )
)

# Distinct LoanIds to join
df_distinct_loans = df_pool_loan.select("LoanId").distinct().filter(F.col("LoanId").isNotNull())

df_next = (
    df_distinct_loans.alias("loan")
    .join(
        df_payment_schedule.alias("ps"),
        F.col("loan.LoanId") == F.col("ps.ExposureKey"),
        "inner"
    )
    .filter(
        (F.col("ps.InstallmentDate").cast("date").between(
            F.lit(start_dt), F.last_day(F.lit(start_dt))
        )) &
        (F.col("ps.InstallmentTypeCode").isin("1", "2", "4"))
    )
    .groupBy(F.col("ps.ExposureKey").alias("nxt_ExposureKey"))
    .agg(
        F.sum("InstallmentAmount").alias("nxt_installmentAmount"),
        F.count(F.lit(1)).alias("nxt_installments"),
        F.max(F.col("InstallmentDate").cast("date")).alias("nxt_installmentdate")
    )
)

# Join installments to pool_loan → final result
df_result = (
    df_pool_loan.alias("pl")
    .join(
        df_next.alias("n"),
        F.col("pl.LoanId") == F.col("n.nxt_ExposureKey"),
        "left"
    )
    .select(
        F.col("pl.platformName"),
        F.col("pl.applicationNumber"),
        F.col("pl.debtNumber"),
        F.col("pl.AccountNumber"),
        F.col("pl.settlementcode"),
        F.col("pl.AssetKey"),
        F.col("pl.LoanId"),
        F.col("pl.isactiveflag"),
        F.col("pl.ReferenceDate"),
        F.col("pl.OutstandingTotalBalance"),
        F.col("pl.Delinquent"),
        F.col("pl.PastDueDays"),
        F.col("pl.BucketId"),
        F.col("pl.LoanStatus"),
        F.col("pl.MaturityDate"),
        F.col("pl.installFrq"),
        F.col("pl.PaymentAmount"),
        F.col("pl.paymentdate"),
        F.col("pl.InterestRate"),
        F.col("pl.accMechanism"),
        F.col("n.nxt_installmentAmount"),
        F.col("n.nxt_installments"),
        F.col("n.nxt_installmentdate")
    )
)

row_count = df_result.count()
if row_count != pool_count:
    raise RuntimeError(f"Final count {row_count:,} != Pool count {pool_count:,}")
print(f"Final result: {row_count:,} rows, {len(df_result.columns)} columns")

In [0]:
if row_count == 0:
    raise RuntimeError("Refusing to write 0 rows")

try:
    df_result.write.format("delta").mode("overwrite").saveAsTable(TARGET_TABLE)
    print(f"Written {row_count:,} rows to {TARGET_TABLE} (base_date={base_date})")
except Exception as e:
    raise RuntimeError(f"Failed to write to {TARGET_TABLE}: {e}") from e